<a href="https://colab.research.google.com/github/maaviarizwan/skin-lesion-research/blob/main/notebooks/Stage2_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Synaptix Research — Skin Lesion Classification
## Stage 2: Preprocessing Pipeline (FIXED — Session-Safe)

```
Stage 1 DONE  Environment + Dataset
Stage 2       Preprocessing Pipeline  <- YOU ARE HERE
Stage 3       Baseline Models (EfficientNet + ViT)
Stage 4       Hybrid Model
Stage 5       Paper Writing
```

---

## Cell 1 — GPU Check + Libraries + Mount Drive

In [ ]:
import torch
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > T4 GPU')

!pip install timm albumentations kaggle --upgrade -q
print('Libraries installed')

from google.colab import drive
drive.mount('/content/drive')
RESEARCH_DIR = '/content/drive/MyDrive/Synaptix_Research'
print(f'Drive mounted. Research folder: {RESEARCH_DIR}')

In [ ]:
import os, warnings, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from collections import Counter
warnings.filterwarnings('ignore')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_SIZE   = 224
BATCH_SIZE = 32

LESION_NAMES = {
    'nv':'Melanocytic Nevi', 'mel':'Melanoma',
    'bkl':'Benign Keratosis', 'bcc':'Basal Cell Carcinoma',
    'akiec':'Actinic Keratoses', 'vasc':'Vascular Lesions', 'df':'Dermatofibroma'
}
print(f'Device: {device}')
print('All imports done')

## Cell 3 — Download Dataset

**Colab resets `/content/` every session.** This cell always re-downloads the dataset when needed. Paste your Kaggle token below.

In [ ]:
import os
import kagglehub

# Install kagglehub
!pip install kagglehub -q

# ── Paste your NEW token here (regenerate it on Kaggle first) ──
os.environ['KAGGLE_API_TOKEN'] = 'KGAT_c7798b0fe3d02db26a83507e2c7e334e'

# ── Download HAM10000 ──────────────────────────────────────────
print('Downloading HAM10000...')
path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")
print(f'Downloaded to: {path}')

# ── Find ALL images (works wherever kagglehub puts them) ───────
image_path_dict = {}
for root, dirs, files in os.walk(path):
    for filename in files:
        if filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            img_id = os.path.splitext(filename)[0]
            image_path_dict[img_id] = os.path.join(root, filename)

print(f'Images found: {len(image_path_dict)}')
if len(image_path_dict) == 0:
    print('ERROR: 0 images. Paste the output of the line above and send it.')
else:
    print('Dataset ready — continue to Cell 4')

## Cell 4 — Load Metadata + Remap Image Paths

Loads the processed CSV saved in Stage 1 and maps each row to its image file.

In [ ]:
df = pd.read_csv(f'{RESEARCH_DIR}/ham10000_processed.csv')

label_map   = {cls: idx for idx, cls in enumerate(sorted(df['dx'].unique()))}
NUM_CLASSES = len(label_map)
idx_to_cls  = {v: k for k, v in label_map.items()}

# Remap paths using the images found in Cell 3
df['path'] = df['image_id'].map(image_path_dict)
df = df[df['path'].notna()].reset_index(drop=True)

print(f'Rows loaded   : {len(df)}')
print(f'Classes       : {NUM_CLASSES}')
print(f'Missing paths : {df["path"].isna().sum()}')

if len(df) < 9000:
    print('WARNING: Less than 9000 images mapped. Rerun Cell 3.')
else:
    print('All paths mapped correctly')

## Cell 5 — Train / Val / Test Split

Stratified split preserves class ratios across all three sets. Split: **70% Train | 15% Val | 15% Test**

In [ ]:
train_val_df, test_df = train_test_split(
    df, test_size=0.15, stratify=df['label'], random_state=SEED
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.176, stratify=train_val_df['label'], random_state=SEED
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train : {len(train_df)} ({len(train_df)/len(df)*100:.1f}%)')
print(f'Val   : {len(val_df)} ({len(val_df)/len(df)*100:.1f}%)')
print(f'Test  : {len(test_df)} ({len(test_df)/len(df)*100:.1f}%)')

## Cell 6 — Fix Class Imbalance

Two tools working together:
- **Class weights** → loss function penalises rare class mistakes more
- **WeightedRandomSampler** → rare classes appear more often each epoch

In [ ]:
class_counts = train_df['label'].value_counts().sort_index().values
class_weights = len(train_df) / (NUM_CLASSES * class_counts)
class_weights_tensor = torch.FloatTensor(class_weights).to(device)

print('Class weights (higher = rarer class):')
for idx, w in enumerate(class_weights):
    cls = idx_to_cls[idx]
    bar = '|' * int(w * 5)
    print(f'  {cls:6} w={w:.3f} {bar}')

sample_weights = torch.DoubleTensor(
    [class_weights[label] for label in train_df['label']]
)
sampler = WeightedRandomSampler(
    weights=sample_weights, num_samples=len(train_df), replacement=True
)
print('\nWeightedRandomSampler created')

## Cell 7 — Augmentation Pipelines

Train: strong augmentation. Val/Test: only resize + normalise.
Using ImageNet stats because pretrained models expect them.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.4),
    A.GridDistortion(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ToTensorV2()
])
print('Augmentation pipelines ready')

## Cell 8 — Custom PyTorch Dataset

In [ ]:
class HAM10000Dataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df        = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        image = np.array(Image.open(row['path']).convert('RGB'))
        label = int(row['label'])
        if self.transform:
            image = self.transform(image=image)['image']
        return image, label

train_dataset = HAM10000Dataset(train_df, transform=train_transform)
val_dataset   = HAM10000Dataset(val_df,   transform=val_test_transform)
test_dataset  = HAM10000Dataset(test_df,  transform=val_test_transform)

img, lbl = train_dataset[0]
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)} | Test: {len(test_dataset)}')
print(f'Image tensor shape : {img.shape}')  # must be [3, 224, 224]
print(f'Pixel range        : [{img.min():.2f}, {img.max():.2f}]')
print(f'Label              : {lbl} ({idx_to_cls[lbl]})')

## Cell 9 — DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')
print(f'Test batches  : {len(test_loader)}')

## Cell 10 — Verify: Visualise Augmented Batch

Always check your pipeline visually before training. Images should look natural but varied.

In [ ]:
def denormalize(tensor):
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)
    return (tensor * std + mean).clamp(0, 1)

images, labels = next(iter(train_loader))

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
fig.suptitle('Augmented Training Batch (32 images)', fontsize=13, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    img = denormalize(images[i]).permute(1,2,0).numpy()
    ax.imshow(img)
    ax.set_title(idx_to_cls[labels[i].item()], fontsize=7)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{RESEARCH_DIR}/figures/augmented_batch.png', dpi=120, bbox_inches='tight')
plt.show()
print('If images look natural — pipeline is working!')

## Cell 11 — Verify: Sampler Balance Check

Confirms WeightedRandomSampler is oversampling rare classes.

In [ ]:
label_counts_sampled  = Counter()
label_counts_original = train_df['dx'].value_counts().to_dict()

for _, labels_batch in train_loader:
    for lbl in labels_batch.numpy():
        label_counts_sampled[idx_to_cls[lbl]] += 1

print(f'{"Class":<8} {"Original":>10} {"Sampled":>10} {"Change":>10}')
print('-' * 42)
for cls in sorted(label_counts_sampled):
    orig    = label_counts_original.get(cls, 0)
    sampled = label_counts_sampled[cls]
    change  = (sampled - orig) / orig * 100
    print(f'{cls:<8} {orig:>10} {sampled:>10} {change:>+9.1f}%')
print('\nRare classes should show positive change')

## Cell 12 — Save Splits + Config to Drive

In [ ]:
train_df.to_csv(f'{RESEARCH_DIR}/split_train.csv', index=False)
val_df.to_csv(f'{RESEARCH_DIR}/split_val.csv',     index=False)
test_df.to_csv(f'{RESEARCH_DIR}/split_test.csv',   index=False)

config = {
    'seed': SEED, 'img_size': IMG_SIZE, 'batch_size': BATCH_SIZE,
    'num_classes': NUM_CLASSES, 'label_map': label_map,
    'class_weights': class_weights.tolist(),
    'imagenet_mean': IMAGENET_MEAN, 'imagenet_std': IMAGENET_STD,
    'train_size': len(train_df), 'val_size': len(val_df), 'test_size': len(test_df)
}
with open(f'{RESEARCH_DIR}/config.json', 'w') as f:
    json.dump(config, f, indent=2)

print('Saved to Drive:')
print(f'  split_train.csv  ({len(train_df)} rows)')
print(f'  split_val.csv    ({len(val_df)} rows)')
print(f'  split_test.csv   ({len(test_df)} rows)')
print(f'  config.json')

## Cell 13 — Stage 2 Complete

In [ ]:
print('=' * 55)
print('  STAGE 2 COMPLETE')
print('=' * 55)
print(f'  Train / Val / Test  : {len(train_df)} / {len(val_df)} / {len(test_df)}')
print(f'  Image size          : {IMG_SIZE}x{IMG_SIZE}')
print(f'  Batch size          : {BATCH_SIZE}')
print(f'  Imbalance fix       : WeightedRandomSampler + Weighted Loss')
print(f'  Augmentations       : 9 transforms on train only')
print(f'  Normalisation       : ImageNet mean/std')
print()
print('  NEXT: Stage 3 — Baseline Models (EfficientNet + ViT)')
print('=' * 55)